<div dir="rtl">

# ✂️ 05 - End-to-End Complete Text Splitting Pipeline (من الصفر للاحتراف)

## لماذا تُعد مرحلة تقسيم النصوص (Chunking) هي الأهم في الـ RAG؟
1. **حدود السياق (Context Window Limits)**: نماذج التضمين واللغة لا تستوعب مستندات كاملة دفعة واحدة.
2. **التركيز الدلالي (Semantic Coherence)**: القطع الصغيرة والمركزة تُنتج متجهات تضمين أعلى جودة وأكثر دقة في البحث.
3. **توفير التكلفة وزمن الاستجابة**: إرسال القطع ذات الصلة فقط للـ LLM بدلاً من إرسال مستندات ضخمة مليئة بالتفاصيل غير المهمة.

---

### 🎯 ما سنتعلمه ونطبقه في هذا الكراس:
1. **استيراد مستندات متنوعة** (نصوص طويلة، مقالات HTML، وبيانات JSON).
2. **تطبيق `RecursiveCharacterTextSplitter`** مع ضبط ذكي للفواصل (`chunk_size` و `chunk_overlap`).
3. **تطبيق `CharacterTextSplitter`** لمقارنة الأداء وسلوك التقسيم.
4. **تطبيق `HTMLHeaderTextSplitter`** للحفاظ على تراتبية وهيكل صفحات الويب.
5. **تطبيق `RecursiveJsonSplitter`** لمعالجة بيانات الـ API والـ JSON دون كسر بنيتها.
6. **قياس إحصائيات الجودة (Chunk Quality Analytics)**: أطوال القطع، نسبة التداخل، والتحقق من انتقال الميتاداتا.

</div>


<div dir="rtl">

### 1️⃣ تجهيز البيئة واستيراد مكتبات التقسيم

</div>


In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    HTMLHeaderTextSplitter,
    RecursiveJsonSplitter
)

load_dotenv(find_dotenv())

# تحديد مسار ملف sample.txt
DATA_DIR = Path("data") if Path("data").exists() else Path("../../data") if Path("../../data").exists() else Path("../data")
sample_text_path = DATA_DIR / "sample.txt"

with open(sample_text_path, "r", encoding="utf-8") as f:
    raw_sample_text = f.read()

print(f"✅ تم تحميل النص التجريبي. الحجم: {len(raw_sample_text):,} حرف.")


<div dir="rtl">

### 2️⃣ التقسيم التكراري الذكي (`RecursiveCharacterTextSplitter`)
الخيار الافتراضي الأفضل لمختلف النصوص؛ يقوم بمحاولة التقسيم عند الفقرات `

`، ثم الجمل `
`، ثم الكلمات ` `، ثم الأحرف.

</div>


In [ ]:
# إعداد المقسم الذكي
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["

", "
", ". ", " ", ""]
)

# إنشاء Document أصلي مع ميتاداتا
doc_obj = Document(
    page_content=raw_sample_text,
    metadata={"source": "sample.txt", "type": "knowledge_base"}
)

recursive_chunks = recursive_splitter.split_documents([doc_obj])

print(f"📊 عدد القطع الناتجة: {len(recursive_chunks)}")
print("\n--- معاينة القطعة الأولى ---")
print(recursive_chunks[0].page_content)
print(f"الميتاداتا المنقولة: {recursive_chunks[0].metadata}")
print(f"طول القطعة: {len(recursive_chunks[0].page_content)} حرف.")


<div dir="rtl">

### 3️⃣ مقارنة مع التقسيم المباشر (`CharacterTextSplitter`)
يقوم بالتقسيم الصارم فقط عند الفاصل المحدد دون التدرج.

</div>


In [ ]:
char_splitter = CharacterTextSplitter(
    separator="

",
    chunk_size=400,
    chunk_overlap=50,
    is_separator_regex=False
)

char_chunks = char_splitter.split_documents([doc_obj])

print(f"📊 عدد القطع الناتجة عبر CharacterTextSplitter: {len(char_chunks)}")
print("💡 نلاحظ أن RecursiveCharacterTextSplitter يحافظ على حدود الحجم أفضل بكثير عند وجود فقرات طويلة.")


<div dir="rtl">

### 4️⃣ تقسيم صفحات الويب الهيكلية (`HTMLHeaderTextSplitter`)
يحافظ على ربط كل فقرة بالعناوين الرئيسية (`H1`, `H2`, `H3`) التي تنتمي إليها كبيانات وصفية.

</div>


In [ ]:
sample_html = """
<!DOCTYPE html>
<html>
<body>
    <h1>LangChain Architecture</h1>
    <p>LangChain is a framework for developing applications powered by language models.</p>
    <h2>Core Components</h2>
    <h3>Document Loaders</h3>
    <p>Document loaders ingest data from diverse sources including PDF, TXT, and Web.</p>
    <h3>Vector Stores</h3>
    <p>Vector stores index embeddings and provide similarity search mechanisms like FAISS and Chroma.</p>
    <h2>Retrieval Strategies</h2>
    <p>Retrievers like MMR provide diverse, non-redundant contextual chunks for RAG.</p>
</body>
</html>
"""

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
html_chunks = html_splitter.split_text(sample_html)

print(f"✅ تم تقسيم الـ HTML إلى {len(html_chunks)} قطعة هيكلية:\n")
for i, chunk in enumerate(html_chunks, 1):
    print(f"--- القطعة {i} ---")
    print(f"المحتوى: {chunk.page_content}")
    print(f"الميتاداتا المرتبطة بالتراتبية: {chunk.metadata}\n")


<div dir="rtl">

### 5️⃣ تقسيم البيانات الهيكلية والـ APIs (`RecursiveJsonSplitter`)
تقسيم كائنات الـ JSON المعقدة والمتداخلة دون إفساد بنية الكائنات والقواميس.

</div>


In [ ]:
sample_json_data = {
    "framework": "LangChain",
    "version": "0.3+",
    "modules": [
        {"name": "Loaders", "description": "Loads data from text, pdf, web", "status": "active"},
        {"name": "Splitters", "description": "Recursive and character splitters", "status": "active"},
        {"name": "Embeddings", "description": "HuggingFace and FastEmbed models", "status": "active"},
        {"name": "VectorStores", "description": "FAISS, Chroma, and Qdrant engines", "status": "active"}
    ],
    "support": {
        "languages": ["Python", "JavaScript"],
        "license": "MIT"
    }
}

json_splitter = RecursiveJsonSplitter(max_chunk_size=150)
json_chunks = json_splitter.split_json(json_data=sample_json_data)

print(f"✅ تم تقسيم الـ JSON إلى {len(json_chunks)} قطعة صغيرة صالحة هيكلياً:\n")
for i, chunk in enumerate(json_chunks, 1):
    print(f"القطعة {i}: {json.dumps(chunk, ensure_ascii=False)}")


<div dir="rtl">

### 6️⃣ تحليل جودة القطع (Chunk Quality Analytics)
فحص إحصائي شامل لتوزيع أطوال القطع، والتحقق من أن جميع القطع جاهزة لنموذج التضمين.

</div>


In [ ]:
chunk_lengths = [len(c.page_content) for c in recursive_chunks]

min_len = min(chunk_lengths)
max_len = max(chunk_lengths)
avg_len = sum(chunk_lengths) / len(chunk_lengths)

print("="*50)
print("📊 تقرير إحصائيات القطع (Recursive Chunks Quality):")
print("="*50)
print(f"• إجمالي عدد القطع: {len(recursive_chunks)}")
print(f"• أقل طول قطعة: {min_len} حرف")
print(f"• أقصى طول قطعة: {max_len} حرف")
print(f"• متوسط طول القطعة: {avg_len:.1f} حرف")
print(f"• جميع القطع تحمل الميتاداتا الأصلية بنجاح: {all('source' in c.metadata for c in recursive_chunks)}")
print("="*50)
print("🎉 اكتملت مرحلة التقسيم بنجاح، والنصوص جاهزة لمرحلة التضمين والتخزين المتجهي!")
